# Pelajaran 11 - Protokol Agen-ke-Agen (A2A)


## Pengaturan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Apa itu Protokol A2A?

Protokol **Agent-to-Agent (A2A) protokol** adalah standar terbuka yang memungkinkan agen AI untuk berkomunikasi,
menemukan satu sama lain, dan berkolaborasi — bahkan ketika mereka dibangun di atas kerangka kerja yang berbeda atau dihosting
oleh layanan yang berbeda.

Konsep utama:

- **Discovery** – Agen menerbitkan sebuah *Kartu Agen* yang menggambarkan kemampuan mereka, sehingga
  mudah bagi agen lain (atau pengatur) untuk menemukan spesialis yang tepat untuk sebuah tugas.
- **Message Passing** – Agen saling bertukar pesan terstruktur melalui protokol umum, sehingga sebuah
  permintaan dari satu agen dapat dipahami dan dipenuhi oleh agen lain terlepas dari implementasi internalnya.
- **Task Lifecycle** – A2A mendefinisikan status seperti *diajukan*, *sedang dikerjakan*, *selesai*, dan
  *gagal*, memberi pengatur (orchestrator) visibilitas penuh tentang bagaimana tugas yang didelegasikan sedang berkembang.

Dalam pelajaran ini kami mensimulasikan kolaborasi bergaya A2A dengan merangkai tiga agen perjalanan khusus
ke dalam alur kerja di mana setiap agen menyumbangkan keahliannya dan meneruskan hasil kepada agen berikutnya.


## Membuat Agen Perjalanan Khusus


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Kolaborasi Multi-Agen melalui Alur Kerja

Kami menghubungkan ketiga agen ke dalam alur kerja berurutan yang mencerminkan pertukaran pesan A2A:

1. **CurrencyExchangeAgent** menerima permintaan pengguna dan menghasilkan panduan mata uang.
2. **ActivityPlannerAgent** menerima konteks yang diperkaya dan menambahkan rekomendasi aktivitas.
3. **TravelManagerAgent** mensintesis kedua masukan menjadi ringkasan perjalanan akhir.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Memahami A2A dalam Produksi

Di lingkungan produksi, protokol A2A membuka skenario lintas-layanan yang kuat:

| Kemampuan | Deskripsi |
|---|---|
| **Interoperabilitas lintas-kerangka** | Agen yang dibangun dengan satu kerangka dapat mendelegasikan tugas ke agen yang dibangun dengan kerangka lain yang sesuai dengan A2A, memungkinkan interoperabilitas lintas organisasi yang sejati. |
| **Batas layanan** | Agen dapat berada di mikrolayanan terpisah, wilayah cloud, atau bahkan organisasi yang berbeda sekalipun sambil tetap berkolaborasi dengan mulus. |
| **Penemuan dinamis** | Sebuah orchestrator dapat menanyakan registri Agent Card pada runtime untuk menemukan spesialis yang paling sesuai untuk sub-tugas tertentu. |
| **Streaming & notifikasi push** | A2A mendukung Server-Sent Events (SSE) untuk pembaruan kemajuan waktu nyata dan notifikasi push untuk tugas yang berjalan lama. |

Alur kerja yang kami bangun di atas adalah versi yang disederhanakan dan berjalan di dalam-proses dari pola ini. Dalam penyebaran nyata, setiap agen akan mengekspos endpoint HTTP, menerbitkan Agent Card, dan berkomunikasi melalui protokol A2A JSON-RPC.


## Ringkasan

Dalam pelajaran ini Anda mempelajari:

1. **Apa itu protokol A2A** — standar terbuka untuk penemuan antar-agen, pengiriman pesan,
   dan manajemen tugas.
2. **Bagaimana membuat agen khusus** — agen Penukaran Mata Uang, agen Perencana Aktivitas,
   dan pengelola Perjalanan (Travel Manager) sebagai orchestrator.
3. **Cara menghubungkan agen ke dalam workflow** — menggunakan `WorkflowBuilder` untuk memodelkan pengiriman pesan berurutan
   antar agen.
4. **Bagaimana A2A bekerja di produksi** — memungkinkan kolaborasi lintas-framework, lintas-layanan dengan penemuan dinamis dan pembaruan streaming.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Penafian:
Dokumen ini diterjemahkan menggunakan layanan terjemahan AI Co-op Translator (https://github.com/Azure/co-op-translator). Meskipun kami berupaya mencapai ketepatan, harap diketahui bahwa terjemahan otomatis dapat mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang otoritatif. Untuk informasi yang bersifat kritis, disarankan menggunakan terjemahan profesional oleh penerjemah manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau salah tafsir yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
